# Machine Learning and Edge AI Workshop

### Step 0: Connect to a runtime and change the runtime type to **"T4 GPU"**

Click the down-arrow button in the top-right corner, as shown in the screenshots below.

> **Important:** Please make sure you select **T4 GPU**, as using a different runtime may lead to much longer execution times.

<img src="https://github.com/YangLi309/CSSE4011-ML-Workshop/blob/main/screenshots/change_runtime_type.png?raw=true" width="300"> &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;
<img src="https://github.com/YangLi309/CSSE4011-ML-Workshop/blob/main/screenshots/runtime_type.png?raw=true" width="300">

## 1. Install Required Packages and Upload MNIST Data

In [1]:
!git clone https://github.com/YangLi309/CSSE4011-ML-Workshop.git
%cd CSSE4011-ML-Workshop/

Cloning into 'CSSE4011-ML-Workshop'...
remote: Enumerating objects: 70035, done.
remote: Counting objects: 100% (70035/70035), done.
remote: Compressing objects: 100% (70033/70033), done.
remote: Total 70035 (delta 0), reused 70025 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (70035/70035), 19.21 MiB | 19.77 MiB/s, done.
Updating files: 100% (70003/70003), done.
/content/CSSE4011-ML-Workshop


In [13]:
# Install ONNX for model export
!pip3 install onnx onnxscript onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.1/164.1 kB 19.8 MB/s eta 0:00:00



## 2. Introduction

### What is PyTorch?
PyTorch is an open-source deep learning framework known for its flexibility and ease of use, making it ideal for rapid prototyping and research.

### What is ONNX?
ONNX (Open Neural Network Exchange) is an open standard format for representing machine learning models, enabling interoperability between various frameworks and devices.

### Why are these important for Edge AI?
- **PyTorch**: Facilitates easy model development and training.
- **ONNX**: Allows exporting models to a standardized format for deployment across different platforms, including resource-constrained edge devices.

### What You Will Learn in This Workshop
- Load and adapt a pretrained model for a new task.
- Evaluate model performance.
- Prune the model to improve inference time.
- Export the model to ONNX format.

## 3. Load Pretrained AlexNet and Adapte the Model for MNIST 10

*   List item
*   List item

Classes

In [2]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torchvision.models import alexnet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Current device:", device)

# Load the original pretrained AlexNet
original_model = alexnet(pretrained=True)

# Now modify for MNIST classification (10 classes)
model = alexnet(pretrained=True)
model.classifier[6] = nn.Linear(4096, 10)  # For 10 classes
model = model.to(device)

Current device: cuda


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:01<00:00, 143MB/s]


## 4. Evaluate the Pretrained AlexNet on MNIST

In [3]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torchvision.models import alexnet
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
import numpy as np

# Transform for MNIST images (which are 28x28 grayscale)
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize to AlexNet input size
    transforms.Grayscale(num_output_channels=3),  # Convert to 3 channels
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load MNIST test data
test_dataset = ImageFolder(root='./data/test', transform=transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Evaluate the model (with modified classifier but before fine-tuning)
correct = 0
total = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Accuracy of AlexNet with modified classifier (before fine-tuning) on MNIST test data: {100 * correct / total:.2f}%")
print("Note: This accuracy is expected to be low as the model hasn't been finetuned for MNIST yet")

# Sample predictions
print("\nSample predictions:")
for i in range(min(10, len(all_preds))):
    print(f"True: {all_labels[i]}, Predicted: {all_preds[i]}")

Accuracy of AlexNet with modified classifier (before fine-tuning) on MNIST test data: 7.02%
Note: This accuracy is expected to be low as the model hasn't been finetuned for MNIST yet

Sample predictions:
True: 0, Predicted: 7
True: 0, Predicted: 7
True: 0, Predicted: 7
True: 0, Predicted: 7
True: 0, Predicted: 5
True: 0, Predicted: 5
True: 0, Predicted: 6
True: 0, Predicted: 7
True: 0, Predicted: 7
True: 0, Predicted: 6


## 5. Freeze All Layers Except Classifier

In [4]:
for param in model.features.parameters():
    param.requires_grad = False

for param in model.classifier.parameters():
    param.requires_grad = True

## 6. Load MNIST Train Dataset

In [5]:
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize to AlexNet input size
    transforms.Grayscale(num_output_channels=3),  # Convert to 3 channels
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Assuming your dataset is stored in "./data/" with subfolders 0/, 1/, ..., 9/
train_dataset = ImageFolder(root='./data/train', transform=transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

## 7. Finetune the Classifier on MNIST

### Training epochs and corresponding model accuracy

- **1 epoch:** 96.45% accuracy  
- **5 epochs:** 97.02% accuracy  
- **10 epochs:** 99.30% accuracy  

> **Note:** The default setting is **1 epoch** for time efficiency, but you may increase the number of epochs to achieve better performance.

> **Reference:** Training for **1 epoch** should take approximately **3 minutes**. If it takes significantly longer, check whether your runtime type is set to **CPU** instead of **GPU**.

In [6]:
import torch.optim as optim
from tqdm import tqdm

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

# You can increase the number of epochs if you want to have higher accuracy
num_epochs = 1

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    # Wrap train_loader with tqdm for progress bar
    train_loader_tqdm = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}')

    for images, labels in train_loader_tqdm:
        images, labels = images.to(device), labels.to(device)

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Statistics
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        # Update progress bar
        train_loader_tqdm.set_postfix({
            'loss': running_loss/total,
            'acc': 100.*correct/total
        })

    # Print epoch statistics
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100. * correct / total
    print(f'\nEpoch {epoch+1}/{num_epochs} - Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.2f}%')

Epoch 1/1: 100%|██████████| 1875/1875 [03:12<00:00,  9.74it/s, loss=0.00358, acc=96.5]


Epoch 1/1 - Loss: 0.1146, Accuracy: 96.47%


## 8. Evaluate the Finetuned Model

In [7]:
# Evaluate the model (with modified classifier but before fine-tuning)
correct = 0
total = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Accuracy of AlexNet with modified classifier (after fine-tuning) on MNIST test data: {100 * correct / total:.2f}%")

# Sample predictions
print("\nSample predictions:")
for i in range(min(10, len(all_preds))):
    print(f"True: {all_labels[i]}, Predicted: {all_preds[i]}")

Accuracy of AlexNet with modified classifier (after fine-tuning) on MNIST test data: 98.42%

Sample predictions:
True: 0, Predicted: 0
True: 0, Predicted: 0
True: 0, Predicted: 0
True: 0, Predicted: 0
True: 0, Predicted: 0
True: 0, Predicted: 0
True: 0, Predicted: 0
True: 0, Predicted: 0
True: 0, Predicted: 0
True: 0, Predicted: 0


## 9. Apply Pruning (50% on Conv Layers)

In [8]:
import torch.nn as nn
from torch.nn.utils import prune
import copy

def prune_model(model):
    # Create a deep copy of the model to avoid modifying the original
    pruned_model = copy.deepcopy(model)

    # Apply pruning to the copied model
    # For AlexNet, we should focus on pruning the convolutional layers
    # The instruction mentioned 50% on Conv layers, so we'll focus on those
    for name, module in pruned_model.named_modules():
        if isinstance(module, nn.Conv2d):
            prune.l1_unstructured(module, name="weight", amount=0.5)
        # We can apply a lighter pruning to fully connected layers
        elif isinstance(module, nn.Linear):
            prune.l1_unstructured(module, name="weight", amount=0.3)

    # Make pruning permanent to ensure the weights are actually removed
    for name, module in pruned_model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)) and hasattr(module, 'weight_orig'):
            prune.remove(module, 'weight')

    return pruned_model

finetuned_model = model
pruned_model = prune_model(model)

## 10. Compare Finetuned Model and Pruned Model

In [9]:
# Evaluate both models and compare accuracy and performance
import time

def evaluate_model(model, dataloader, device):
    model.eval()
    correct = 0
    total = 0
    inference_times = []

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)

            # Measure inference time
            start_time = time.time()
            outputs = model(images)
            end_time = time.time()

            inference_time = end_time - start_time
            inference_times.append(inference_time)

            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    avg_inference_time = sum(inference_times) / len(inference_times)
    fps = 1.0 / (avg_inference_time / images.size(0))  # FPS = batch_size / avg_time_per_batch

    return accuracy, fps

# Evaluate the finetuned model
print("Evaluating finetuned model...")
finetuned_accuracy, finetuned_fps = evaluate_model(finetuned_model, test_loader, device)

# Evaluate the pruned model
print("Evaluating pruned model...")
pruned_accuracy, pruned_fps = evaluate_model(pruned_model, test_loader, device)

# Print comparison results
print("\n----- Model Comparison -----")
print(f"{'Model':<20} {'Accuracy':<15} {'FPS':<10}")
print("-" * 45)
print(f"{'Finetuned':<20} {finetuned_accuracy:.2f}%{'':<10} {finetuned_fps:.2f}")
print(f"{'Pruned (50%)':<20} {pruned_accuracy:.2f}%{'':<10} {pruned_fps:.2f}")
print("-" * 45)

# Calculate differences
acc_diff = pruned_accuracy - finetuned_accuracy
fps_diff = pruned_fps - finetuned_fps
fps_improvement = (fps_diff / finetuned_fps) * 100

print(f"\nAccuracy change after pruning: {acc_diff:.2f}%")
print(f"FPS improvement after pruning: {fps_improvement:.2f}% ({fps_diff:.2f} FPS)")
# Note on performance improvements
print("\nNote: The model inference speed improvement here is not that obvious due to the limit of the I/O speed.")
print("When it comes to a bigger model and higher resolution images, the difference will be more obvious.")
print("Factors such as memory bandwidth, disk I/O, and data preprocessing can mask the actual computational benefits of pruning.")
print("In real-world deployment scenarios with larger models (e.g., ResNet, EfficientNet) and higher resolution images,")
print("the performance gains from pruning would be more significant and noticeable.")

Evaluating finetuned model...
Evaluating pruned model...

----- Model Comparison -----
Model                Accuracy        FPS       
---------------------------------------------
Finetuned            98.87%           12648.61
Pruned (50%)         89.34%           13133.63
---------------------------------------------

Accuracy change after pruning: -9.53%
FPS improvement after pruning: 3.83% (485.02 FPS)

Note: The model inference speed improvement here is not that obvious due to the limit of the I/O speed.
When it comes to a bigger model and higher resolution images, the difference will be more obvious.
Factors such as memory bandwidth, disk I/O, and data preprocessing can mask the actual computational benefits of pruning.
In real-world deployment scenarios with larger models (e.g., ResNet, EfficientNet) and higher resolution images,
the performance gains from pruning would be more significant and noticeable.


## 11. Export Finetuned Model and Pruned Model to ONNX

In [15]:
dummy_input = torch.randn(1, 3, 224, 224).to(device)

# Finetuned model
torch.onnx.export(finetuned_model, dummy_input, "../alexnet_mnist.onnx", opset_version=18)

# Pruned
torch.onnx.export(pruned_model, dummy_input, "../alexnet_mnist_pruned.onnx", opset_version=18)

W0413 13:44:50.341000 1752 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0413 13:44:50.342000 1752 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0413 13:44:50.345000 1752 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
W0413 13:44:50.347000 1752 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.


[torch.onnx] Obtain model graph for `AlexNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `AlexNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


W0413 13:44:52.256000 1752 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0413 13:44:52.258000 1752 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0413 13:44:52.260000 1752 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
W0413 13:44:52.262000 1752 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.


[torch.onnx] Obtain model graph for `AlexNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `AlexNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


ONNXProgram(
    model=
        <
            ir_version=10,
            opset_imports={'': 18},
            producer_name='pytorch',
            producer_version='2.10.0+cu128',
            domain=None,
            model_version=None,
        >
        graph(
            name=main_graph,
            inputs=(
                %"x"<FLOAT,[1,3,224,224]>
            ),
            outputs=(
                %"linear_2"<FLOAT,[1,10]>
            ),
            initializers=(
                %"features.0.bias"<FLOAT,[64]>{TorchTensor(...)},
                %"features.0.weight"<FLOAT,[64,3,11,11]>{TorchTensor(...)},
                %"features.3.bias"<FLOAT,[192]>{TorchTensor(...)},
                %"features.3.weight"<FLOAT,[192,64,5,5]>{TorchTensor(...)},
                %"features.6.bias"<FLOAT,[384]>{TorchTensor(...)},
                %"features.6.weight"<FLOAT,[384,192,3,3]>{TorchTensor(...)},
                %"features.8.bias"<FLOAT,[256]>{TorchTensor(...)},
                %"features.8.w

In [18]:
onnx_program = torch.onnx.export(
    finetuned_model,
    (dummy_input,),
    dynamo=True
)
onnx_program.save("../alexnet_mnist.onnx")

W0413 13:49:34.568000 1752 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0413 13:49:34.570000 1752 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0413 13:49:34.572000 1752 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
W0413 13:49:34.574000 1752 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.


[torch.onnx] Obtain model graph for `AlexNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `AlexNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


In [20]:
import numpy as np
import onnxruntime as ort
import torch

finetuned_model.eval()

device = next(finetuned_model.parameters()).device
dummy_input = torch.randn(1, 3, 224, 224).to(device)

with torch.no_grad():
    torch_out = finetuned_model(dummy_input).cpu().numpy()

session = ort.InferenceSession("../alexnet_mnist.onnx")
input_name = session.get_inputs()[0].name
onnx_out = session.run(None, {input_name: dummy_input.cpu().numpy()})[0]

print("Export/inference OK:", np.allclose(torch_out, onnx_out, atol=1e-4))
print("Max abs diff:", np.max(np.abs(torch_out - onnx_out)))

Export/inference OK: True
Max abs diff: 9.536743e-07


## 12. Recap


You now have two ONNX models ready for deployment:
- `alexnet_mnist.onnx`
- `alexnet_mnist_pruned.onnx`
